# 00 — Random genomic windows (control set for Evo 2 exploration)

Produces a single file of **2000 random 10 kb windows** from hg19 autosomes + chrX, with no overlap to any MYC or CTCF ChIP-seq peak, with sequences extracted from `hg19.fa`. Output is written to the `extracted_windows/RANDOM/` directory in the same CSV schema that `01_windows.ipynb` produces, so `02_embeddings.ipynb` picks it up unchanged via its existing file discovery.

This notebook replaces the peak → window two-step for the random control. Real ChIP-seq peaks have variable widths and need centring; random windows don't, so routing them through `01_windows` adds a round-trip (midpoint of a 10 kb row gets used to extract another 10 kb) that's easy to get wrong. End-to-end is cleaner here.

Rationale for this control: Block 2 found positional profile differences between MYC and CTCF embeddings, but we can't separate TF-specific biology from generic Evo 2 behavior on DNA without a baseline. This notebook is that baseline.

The ENCODE blacklist is deliberately *not* used here — it covers ~1.4% of the genome and is defined by ChIP-seq artifact behavior, not sequence composition. At 2000 uniformly sampled windows the contamination rate is negligible for file-averaged positional profiles. Blacklist exclusion belongs in the Block 4 classification negative set, not in this descriptive control.

In [1]:
import bisect
from pathlib import Path
from collections import defaultdict

import numpy as np
import polars as pl
import pyfaidx


## Paths and constants

Same path conventions as `01_windows.ipynb`. `WINDOW_LEN = 10000` matches `STUDIED_SPAN` in that notebook and `SEQ_LEN` downstream. Output goes to `extracted_windows/RANDOM/` so it sits alongside the per-TF directories produced by `01_windows`.

In [2]:
# --- paths ---
DATA_ROOT = Path.home() / "vambersky_t/Data"
BED_DIR = DATA_ROOT / "encode_beds"
REF_GENOME = DATA_ROOT / "reference" / "hg19.fa"
FAI_PATH = DATA_ROOT / "reference" / "hg19.fa.fai"
WINDOWS_DIR = DATA_ROOT / "extracted_windows"

# output matches 01_windows naming: {ACCESSION}__{TF}__{BIOSAMPLE}.full_peaks.csv.gz
OUTPUT_DIR = WINDOWS_DIR / "RANDOM"
OUTPUT_PATH = OUTPUT_DIR / "RANDOM__RANDOM__hg19.full_peaks.csv.gz"

# --- parameters ---
WINDOW_LEN = 10_000
HALF_WIN = WINDOW_LEN // 2
N_WINDOWS = 2000
RANDOM_SEED = 42
MAX_ATTEMPTS = 200_000  # hard cap on rejection-sampling attempts

# hg19 autosomes + chrX; chrY and chrM excluded per spec
ALLOWED_CHROMS = [f"chr{i}" for i in range(1, 23)] + ["chrX"]

# TFs whose peaks are excluded from the random set
EXCLUDE_TFS = ["MYC", "CTCF"]

# --- sanity checks ---
assert DATA_ROOT.exists(), f"Data root not found: {DATA_ROOT}"
assert REF_GENOME.exists(), f"Reference genome not found: {REF_GENOME}"
assert FAI_PATH.exists(), f"fai index not found: {FAI_PATH}"
assert BED_DIR.exists(), f"BED directory not found: {BED_DIR}"


## Load chromosome sizes

From `hg19.fa.fai`. Filter to autosomes + chrX. A window is centred on `center`, so it spans `[center - HALF_WIN, center + HALF_WIN)`. For the window to fit entirely within the chromosome, `center` must be in `[HALF_WIN, length - HALF_WIN)`. We precompute the size of that range per chromosome for weighted sampling.

In [3]:
fai = pl.read_csv(
    FAI_PATH,
    separator="\t",
    has_header=False,
    new_columns=["chr", "length", "offset", "linebases", "linewidth"],
)

chrom_sizes = {
    row["chr"]: row["length"]
    for row in fai.iter_rows(named=True)
    if row["chr"] in ALLOWED_CHROMS
}

# usable range for a window centre: [HALF_WIN, length - HALF_WIN)
chrom_usable = {c: l - WINDOW_LEN for c, l in chrom_sizes.items()}
assert all(u > 0 for u in chrom_usable.values()), "some chrom shorter than WINDOW_LEN"

missing = [c for c in ALLOWED_CHROMS if c not in chrom_sizes]
assert not missing, f"missing chromosomes in fai: {missing}"

print(f"Loaded sizes for {len(chrom_sizes)} chromosomes")
print(f"Total usable genome: {sum(chrom_usable.values()):,} bp")


Loaded sizes for 23 chromosomes
Total usable genome: 3,036,073,846 bp


## Load MYC + CTCF exclusion intervals

Concatenate all peaks across all MYC/CTCF narrowPeak files into a single per-chromosome interval set. For each chromosome, store:

- `starts`, `ends`: numpy arrays sorted by `start`
- `cummax_ends`: running maximum of `ends` over the sorted array

The `cummax_ends` array turns overlap-check into two numpy operations. For a candidate window `[qs, qe)`:

1. `k = searchsorted(starts, qe)` — number of intervals with `start < qe`
2. If `k > 0` and `cummax_ends[k-1] > qs`, some interval overlaps

This is the standard interval-stabbing trick. O(log n) per query, and avoids adding `pybedtools` or `pyranges` as dependencies.

In [4]:
bed_files = []
for tf in EXCLUDE_TFS:
    bed_files.extend(sorted(BED_DIR.glob(f"*__{tf}__*.bed.gz")))

print(f"Found {len(bed_files)} exclusion BED files ({EXCLUDE_TFS})")
for f in bed_files:
    print(f"  {f.name}")

assert bed_files, "no MYC/CTCF BED files found — cannot build exclusion set"


Found 17 exclusion BED files (['MYC', 'CTCF'])
  ENCFF043WTJ__MYC__K562.bed.gz
  ENCFF149BIS__MYC__MCF-7.bed.gz
  ENCFF356PWE__MYC__A549.bed.gz
  ENCFF372RQZ__MYC__H1.bed.gz
  ENCFF674RQX__MYC__A549.bed.gz
  ENCFF700CXD__MYC__H1.bed.gz
  ENCFF765CKK__MYC__GM12878.bed.gz
  ENCFF800JFG__MYC__HepG2.bed.gz
  ENCFF017XLW__CTCF__GM12878.bed.gz
  ENCFF123DAC__CTCF__HepG2.bed.gz
  ENCFF252PLM__CTCF__HepG2.bed.gz
  ENCFF536RGD__CTCF__GM12878.bed.gz
  ENCFF692RPA__CTCF__H1.bed.gz
  ENCFF769AUF__CTCF__K562.bed.gz
  ENCFF820BVW__CTCF__MCF-7.bed.gz
  ENCFF962IZR__CTCF__MCF-7.bed.gz
  ENCFF966KGI__CTCF__A549.bed.gz


In [5]:
peaks_per_file = []
for f in bed_files:
    df = pl.read_csv(
        f,
        separator="\t",
        has_header=False,
        comment_prefix="#",
        new_columns=["chr", "start", "end", "name", "score", "strand",
                     "signal_value", "p_value", "q_value", "peak"],
    ).select(["chr", "start", "end"])
    peaks_per_file.append(df)

all_peaks = pl.concat(peaks_per_file).filter(pl.col("chr").is_in(ALLOWED_CHROMS))
print(f"Total exclusion peaks (autosomes + chrX): {len(all_peaks):,}")
print(all_peaks.group_by("chr").len().sort("chr"))


Total exclusion peaks (autosomes + chrX): 507,407
shape: (23, 2)
┌───────┬───────┐
│ chr   ┆ len   │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ chr1  ┆ 46818 │
│ chr10 ┆ 23336 │
│ chr11 ┆ 26567 │
│ chr12 ┆ 24838 │
│ chr13 ┆ 10214 │
│ …     ┆ …     │
│ chr6  ┆ 27331 │
│ chr7  ┆ 26120 │
│ chr8  ┆ 22262 │
│ chr9  ┆ 21583 │
│ chrX  ┆ 13727 │
└───────┴───────┘


In [6]:
# build per-chromosome sorted arrays + cummax_ends
exclusion = {}
for chrom in ALLOWED_CHROMS:
    sub = all_peaks.filter(pl.col("chr") == chrom).sort("start")
    if len(sub) == 0:
        exclusion[chrom] = {
            "starts": np.array([], dtype=np.int64),
            "ends": np.array([], dtype=np.int64),
            "cummax_ends": np.array([], dtype=np.int64),
        }
        continue
    starts = sub["start"].to_numpy().astype(np.int64)
    ends = sub["end"].to_numpy().astype(np.int64)
    exclusion[chrom] = {
        "starts": starts,
        "ends": ends,
        "cummax_ends": np.maximum.accumulate(ends),
    }

total_excl = sum(len(v["starts"]) for v in exclusion.values())
print(f"Built exclusion arrays for {len(exclusion)} chromosomes, {total_excl:,} intervals")


Built exclusion arrays for 23 chromosomes, 507,407 intervals


## Overlap helpers

`overlaps_exclusion` uses the searchsorted + cummax trick on the precomputed exclusion arrays.

`overlaps_accepted` checks the candidate against windows already accepted on the same chromosome. All accepted windows have the same length `WINDOW_LEN`, so we only track starts and use a simple bisect-based neighbour check.

In [7]:
def overlaps_exclusion(chrom: str, qs: int, qe: int) -> bool:
    """True iff [qs, qe) overlaps any interval in the exclusion set on chrom."""
    arr = exclusion[chrom]
    starts = arr["starts"]
    if starts.size == 0:
        return False
    k = int(np.searchsorted(starts, qe, side="left"))
    if k == 0:
        return False
    return bool(arr["cummax_ends"][k - 1] > qs)


def overlaps_accepted(accepted_starts: list, qs: int, qe: int) -> bool:
    """True iff [qs, qe) overlaps any window whose start is in sorted list accepted_starts.

    All accepted windows have length WINDOW_LEN, so we only need starts.
    Overlap with an accepted window at start a means a < qe and a + WINDOW_LEN > qs,
    i.e. a in (qs - WINDOW_LEN, qe).
    """
    if not accepted_starts:
        return False
    lo = bisect.bisect_right(accepted_starts, qs - WINDOW_LEN)
    if lo >= len(accepted_starts):
        return False
    return accepted_starts[lo] < qe


## Rejection sampling

Per attempt:

1. Pick a chromosome with probability proportional to its usable length (`length - WINDOW_LEN`).
2. Pick a uniform centre in `[HALF_WIN, length - HALF_WIN)`.
3. Reject if the window `[center - HALF_WIN, center + HALF_WIN)` overlaps any exclusion interval.
4. Reject if it overlaps an already-accepted window.

Chromosome-bound check is implicit in step 2. `MAX_ATTEMPTS` is a safety cap so the notebook fails loudly if misconfigured.

In [8]:
rng = np.random.default_rng(RANDOM_SEED)

chrom_list = list(chrom_sizes.keys())
weights = np.array([chrom_usable[c] for c in chrom_list], dtype=np.float64)
probs = weights / weights.sum()

# accepted windows: map chrom -> sorted list of window starts
accepted = defaultdict(list)
records = []  # list of dicts with chr, start, end, center

attempts = 0
rej_excl = 0
rej_self = 0

while len(records) < N_WINDOWS and attempts < MAX_ATTEMPTS:
    attempts += 1
    chrom = chrom_list[rng.choice(len(chrom_list), p=probs)]
    center = int(rng.integers(HALF_WIN, chrom_sizes[chrom] - HALF_WIN))
    start = center - HALF_WIN
    end = center + HALF_WIN

    if overlaps_exclusion(chrom, start, end):
        rej_excl += 1
        continue
    if overlaps_accepted(accepted[chrom], start, end):
        rej_self += 1
        continue

    bisect.insort(accepted[chrom], start)
    records.append({"chr": chrom, "start": start, "end": end, "center": center})

assert len(records) == N_WINDOWS, (
    f"only {len(records)} windows accepted after {attempts} attempts — "
    f"MAX_ATTEMPTS too low or exclusions too aggressive"
)

acceptance_rate = len(records) / attempts
print(f"Accepted: {len(records):,}")
print(f"Attempts: {attempts:,}")
print(f"Rejected (exclusion peaks): {rej_excl:,}")
print(f"Rejected (self-overlap):    {rej_self:,}")
print(f"Acceptance rate: {acceptance_rate:.1%}")

if acceptance_rate < 0.10:
    print("\nWARNING: acceptance rate below 10%. "
          "Exclusions may be more aggressive than expected — investigate.")


Accepted: 2,000
Attempts: 2,714
Rejected (exclusion peaks): 700
Rejected (self-overlap):    14
Acceptance rate: 73.7%


## Extract sequences

Same logic as `extract_sequences` in `01_windows.ipynb`: open `hg19.fa` with `pyfaidx`, slice `[start, end)` per record, uppercase, mask non-ACGT to `N`. The bounds check from `01_windows` is redundant here (we already sampled within chromosome bounds) but kept as a defensive assertion.

In [9]:
fasta = pyfaidx.Fasta(str(REF_GENOME))
chrom_sizes_fa = {k: len(v) for k, v in fasta.items()}

for rec in records:
    chrom, start, end = rec["chr"], rec["start"], rec["end"]
    assert 0 <= start and end <= chrom_sizes_fa[chrom], (
        f"window out of bounds: {chrom}:{start}-{end} (chrom size {chrom_sizes_fa[chrom]})"
    )
    seq = str(fasta[chrom][start:end]).upper()
    seq = "".join(b if b in "ACGT" else "N" for b in seq)
    assert len(seq) == WINDOW_LEN, f"sequence length mismatch: {len(seq)} != {WINDOW_LEN}"
    rec["sequence"] = seq

fasta.close()

print(f"Extracted {len(records)} sequences of {WINDOW_LEN} bp each")


Extracted 2000 sequences of 10000 bp each


## Write output

Column order matches what `01_windows.ipynb` produces: `chr, start, end, center, peak_id, sequence`. Here `start` and `end` are the window bounds themselves (so `end - start == WINDOW_LEN`), and `center = start + HALF_WIN`. This is a stricter invariant than real ChIP-seq windows where `start`/`end` are the peak extent, but `02_embeddings` only reads `sequence` (and possibly `peak_id`), so the extra consistency here is harmless.

Saved to `extracted_windows/RANDOM/RANDOM__RANDOM__hg19.full_peaks.csv.gz` — the filename matches the `{accession}__{tf}__{biosample}.full_peaks.csv.gz` pattern that `02_embeddings` globs for.

In [11]:
out_df = pl.DataFrame({
    "chr":      [r["chr"] for r in records],
    "start":    [r["start"] for r in records],
    "end":      [r["end"] for r in records],
    "center":   [r["center"] for r in records],
    "peak_id":  [f"peak_{i}" for i in range(len(records))],
    "sequence": [r["sequence"] for r in records],
})

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# newer polars requires explicit compression when the filename ends in .gz
out_df.write_csv(OUTPUT_PATH, compression="gzip")

print(f"Wrote {len(out_df)} windows to {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size:,} bytes")
print()
print("Output schema:")
print(out_df.schema)
print()
print("First row preview (sequence truncated):")
first = out_df.row(0, named=True)
for k, v in first.items():
    if k == "sequence":
        print(f"  {k}: {v[:60]}... ({len(v)} bp)")
    else:
        print(f"  {k}: {v}")


Wrote 2000 windows to /home/jovyan/vambersky_t/Data/extracted_windows/RANDOM/RANDOM__RANDOM__hg19.full_peaks.csv.gz
File size: 5,164,988 bytes

Output schema:
Schema({'chr': String, 'start': Int64, 'end': Int64, 'center': Int64, 'peak_id': String, 'sequence': String})

First row preview (sequence truncated):
  chr: chr13
  start: 75380376
  end: 75390376
  center: 75385376
  peak_id: peak_0
  sequence: ACTGCCAGTTTTGAGTGGGGTCCAGAACAGAAGATGGCTCTGTAACAGGTCCAGGCTACT... (10000 bp)


## Verification

Six checks, performed by reading the file back so any bug in the write step is caught too:

1. Row count equals `N_WINDOWS`
2. Every row has `end - start == WINDOW_LEN` and `center == start + HALF_WIN`
3. Every sequence has length `WINDOW_LEN` and contains only ACGTN
4. No window overlaps any MYC/CTCF peak (full check)
5. No two windows in the output overlap each other
6. Per-chromosome distribution is roughly proportional to usable length

In [12]:
check_df = pl.read_csv(OUTPUT_PATH)
print(f"Re-read {len(check_df)} rows, columns: {check_df.columns}")

# check 1: row count
assert len(check_df) == N_WINDOWS, f"expected {N_WINDOWS} rows, got {len(check_df)}"
print(f"[OK] Row count: {len(check_df)}")

# check 2: geometry
lengths = (check_df["end"] - check_df["start"]).to_numpy()
assert (lengths == WINDOW_LEN).all(), f"found windows with length != {WINDOW_LEN}"
centers_from_bounds = (check_df["start"] + HALF_WIN).to_numpy()
assert (centers_from_bounds == check_df["center"].to_numpy()).all(), "center != start + HALF_WIN"
print(f"[OK] All windows exactly {WINDOW_LEN} bp, center = start + {HALF_WIN}")


Re-read 2000 rows, columns: ['chr', 'start', 'end', 'center', 'peak_id', 'sequence']
[OK] Row count: 2000
[OK] All windows exactly 10000 bp, center = start + 5000


In [13]:
# check 3: sequence integrity
allowed = set("ACGTN")
bad_seq = []
for row in check_df.iter_rows(named=True):
    s = row["sequence"]
    if len(s) != WINDOW_LEN:
        bad_seq.append((row["peak_id"], f"length {len(s)}"))
        continue
    extra = set(s) - allowed
    if extra:
        bad_seq.append((row["peak_id"], f"unexpected chars {extra}"))
assert not bad_seq, f"{len(bad_seq)} bad sequences, first: {bad_seq[:3]}"
print(f"[OK] All {len(check_df)} sequences are {WINDOW_LEN} bp and ACGTN-only")


[OK] All 2000 sequences are 10000 bp and ACGTN-only


In [14]:
# check 4: no overlap with any MYC/CTCF peak (full check)
bad_excl = []
for row in check_df.iter_rows(named=True):
    if overlaps_exclusion(row["chr"], row["start"], row["end"]):
        bad_excl.append((row["chr"], row["start"], row["end"]))
assert not bad_excl, f"{len(bad_excl)} windows overlap MYC/CTCF peaks, first: {bad_excl[:3]}"
print(f"[OK] No overlap with {total_excl:,} MYC/CTCF peaks (all {len(check_df)} windows checked)")


[OK] No overlap with 507,407 MYC/CTCF peaks (all 2000 windows checked)


In [15]:
# check 5: no two windows in the output overlap each other
conflicts = []
for chrom in ALLOWED_CHROMS:
    sub = check_df.filter(pl.col("chr") == chrom).sort("start")
    starts_arr = sub["start"].to_numpy()
    ends_arr = sub["end"].to_numpy()
    if len(starts_arr) < 2:
        continue
    overlap_mask = starts_arr[1:] < ends_arr[:-1]
    if overlap_mask.any():
        idxs = np.where(overlap_mask)[0]
        for i in idxs:
            conflicts.append((chrom, int(starts_arr[i]), int(ends_arr[i]),
                              int(starts_arr[i + 1]), int(ends_arr[i + 1])))
assert not conflicts, f"{len(conflicts)} self-overlaps, first: {conflicts[:3]}"
print(f"[OK] No self-overlaps among {len(check_df)} windows")


[OK] No self-overlaps among 2000 windows


In [16]:
# check 6: chromosome distribution roughly proportional to usable length
counts = check_df.group_by("chr").len().sort("chr")
total_usable = sum(chrom_usable.values())

print(f"{"chr":<8} {"count":>6} {"expected":>9} {"ratio":>8}")
print("-" * 35)
for chrom in ALLOWED_CHROMS:
    n = counts.filter(pl.col("chr") == chrom)["len"]
    n = int(n[0]) if len(n) else 0
    expected = N_WINDOWS * chrom_usable[chrom] / total_usable
    ratio = n / expected if expected > 0 else float("nan")
    print(f"{chrom:<8} {n:>6d} {expected:>9.1f} {ratio:>8.2f}")


chr       count  expected    ratio
-----------------------------------
chr1        171     164.2     1.04
chr2        176     160.2     1.10
chr3        131     130.4     1.00
chr4        140     125.9     1.11
chr5        120     119.2     1.01
chr6         97     112.7     0.86
chr7        109     104.8     1.04
chr8         96      96.4     1.00
chr9         97      93.0     1.04
chr10        79      89.3     0.88
chr11        84      88.9     0.94
chr12        82      88.2     0.93
chr13        90      75.9     1.19
chr14        74      70.7     1.05
chr15        70      67.5     1.04
chr16        53      59.5     0.89
chr17        39      53.5     0.73
chr18        52      51.4     1.01
chr19        32      38.9     0.82
chr20        28      41.5     0.67
chr21        42      31.7     1.32
chr22        28      33.8     0.83
chrX        110     102.3     1.08


## Done

In [17]:
print("=" * 60)
print("Random control set generated successfully.")
print("=" * 60)
print()
print(f"Output: {OUTPUT_PATH}")
print(f"Windows: {N_WINDOWS} x {WINDOW_LEN} bp, hg19 autosomes + chrX")
print(f"Excluded: all peaks in MYC + CTCF ChIP-seq BED files")
print()
print("Next steps:")
print("  1. Add 'RANDOM' to tfs_to_process.txt in the repo root")
print("  2. Skip 01_windows.ipynb — this notebook already produced the")
print("     extracted_windows output that 01_windows would have written.")
print("  3. Run 02_embeddings.ipynb — picks up RANDOM__RANDOM__hg19 via")
print("     its file discovery and runs Evo 2 inference.")
print("  4. Rerun 03_exploration.ipynb — picks up TF=RANDOM automatically.")


Random control set generated successfully.

Output: /home/jovyan/vambersky_t/Data/extracted_windows/RANDOM/RANDOM__RANDOM__hg19.full_peaks.csv.gz
Windows: 2000 x 10000 bp, hg19 autosomes + chrX
Excluded: all peaks in MYC + CTCF ChIP-seq BED files

Next steps:
  1. Add 'RANDOM' to tfs_to_process.txt in the repo root
  2. Skip 01_windows.ipynb — this notebook already produced the
     extracted_windows output that 01_windows would have written.
  3. Run 02_embeddings.ipynb — picks up RANDOM__RANDOM__hg19 via
     its file discovery and runs Evo 2 inference.
  4. Rerun 03_exploration.ipynb — picks up TF=RANDOM automatically.
